# ⚖️ PakLex AI — Notebook 3: Flask API Server

**Purpose:** Launch the Flask REST API that connects the Dark Judicial frontend to the LangChain + Groq RAG engine.

---
| Endpoint | Method | Description |
|----------|--------|-------------|
| `/` | GET | Serves `index.html` frontend |
| `/api/query` | POST | Runs RAG → returns answer + sources |
| `/api/health` | GET | Checks Groq API + ChromaDB status |
| `/api/stats` | GET | Returns chunk count, model info |

---
> ⚠️ **Prerequisites:**
> - `01_ingest.ipynb` run (vector_store/ must exist)
> - `.env` file with `GROQ_API_KEY=gsk_...`
> - Keep this notebook running while using the app

## Step 1 — Load API Key & Initialise Everything

In [1]:
import os, time
from pathlib import Path
from dotenv import load_dotenv

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain.schema import Document

# ── Load API key from .env (never hardcoded) ───────────────────────────────
load_dotenv('../.env')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

if not GROQ_API_KEY:
    raise ValueError('❌ GROQ_API_KEY missing in ../.env file')
print(f'✅ Groq API key: gsk_...{GROQ_API_KEY[-6:]}')

# ── Config ─────────────────────────────────────────────────────────────────
VECTOR_STORE_DIR = '../vector_store'
COLLECTION_NAME  = 'pakistan_penal_code'
EMBEDDING_MODEL  = 'all-MiniLM-L6-v2'
GROQ_MODEL       = 'llama-3.1-8b-instant'
TEMPLATES_DIR    = '../templates'
TOP_K            = 15

# ── Load embeddings ────────────────────────────────────────────────────────
print('⏳ Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name    = EMBEDDING_MODEL,
    model_kwargs  = {'device': 'cpu'},
    encode_kwargs = {'normalize_embeddings': True},
)
print('✅ Embeddings ready')

# ── Load ChromaDB ──────────────────────────────────────────────────────────
print('⏳ Loading ChromaDB...')
vectordb = Chroma(
    persist_directory  = VECTOR_STORE_DIR,
    embedding_function = embeddings,
    collection_name    = COLLECTION_NAME,
)
vector_retriever = vectordb.as_retriever(
    search_type='mmr', 
    search_kwargs={'k': 10, 'fetch_k': 50}
)

# 2. Extract your saved text chunks directly from ChromaDB
db_data = vectordb.get()
saved_chunks = [
    Document(page_content=t, metadata=m) 
    for t, m in zip(db_data['documents'], db_data['metadatas'])
]

# 3. Build a Keyword Retriever (BM25) for exact text/number matches
bm25_retriever = BM25Retriever.from_documents(saved_chunks)
bm25_retriever.k = 10

# 4. Combine them into a Hybrid Retriever (50% Meaning / 50% Exact Keywords)
retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)
doc_count = vectordb._collection.count()
print(f'✅ ChromaDB ready — {doc_count} chunks')

# ── Load Groq LLM via LangChain ────────────────────────────────────────────
print('⏳ Connecting to Groq API...')
llm = ChatGroq(
    groq_api_key = GROQ_API_KEY,
    model_name   = GROQ_MODEL,
    temperature  = 0.1,
    max_tokens   = 1024,
)
print(f'✅ Groq LLM ready — {GROQ_MODEL}')

# ── Build RAG chain ────────────────────────────────────────────────────────
PROMPT_TEMPLATE = """\
You are PakLex AI, an expert legal assistant for the Pakistan Penal Code (PPC) 1860.

STRICT RULES:
1. Answer ONLY from the CONTEXT provided below.
2. Always cite the Section number (e.g. Section 302) in your answer.
3. If the answer is not in the context, say exactly:
   "I don't have enough information from the Pakistan Penal Code to answer this question."
4. Never hallucinate or use outside knowledge.
5. Be clear, precise, and professional.

=== CONTEXT FROM PAKISTAN PENAL CODE 1860 ===
{context}
=== END CONTEXT ===

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

def format_docs(docs):
    return '\n\n'.join(
        f'[Page {d.metadata.get("page","?")} | {d.metadata.get("source_file","PPC")}]\n{d.page_content}'
        for i, d in enumerate(docs)
    )

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print('\n✅ All components initialised — ready to start server')

KeyboardInterrupt: 

## Step 2 — RAG Query Function

In [ ]:
def rag_query(query: str) -> dict:
    t0   = time.time()
    docs = retriever.get_relevant_documents(query)

    if not docs:
        return {'answer': 'No relevant PPC sections found.', 'sources': [], 'response_time': 0, 'query': query}

    answer  = rag_chain.invoke(query)
    sources = [{
        'content'     : d.page_content,
        'page'        : d.metadata.get('page', ''),
        'source_file' : d.metadata.get('source_file', ''),
        'section_number': d.metadata.get('section_number', ''),
        'section_title' : d.metadata.get('section_title', ''),
        'chapter'       : d.metadata.get('chapter', ''),
    } for d in docs]

    return {
        'answer'       : answer,
        'sources'      : sources,
        'response_time': round(time.time() - t0, 2),
        'query'        : query,
    }


def check_groq() -> str:
    """Ping Groq API to verify connectivity."""
    try:
        llm.invoke('ping')
        return 'ok'
    except:
        return 'error'


print('✅ rag_query() and check_groq() defined')

## Step 3 — Flask App with All API Endpoints

In [ ]:
from flask import Flask, request, jsonify, send_from_directory
from flask_cors import CORS

app = Flask(__name__, template_folder=TEMPLATES_DIR)
CORS(app)


# ── GET / → Serve frontend ─────────────────────────────────────────────────
@app.route('/')
def index():
    return send_from_directory(TEMPLATES_DIR, 'index.html')


# ── POST /api/query ────────────────────────────────────────────────────────
@app.route('/api/query', methods=['POST'])
def api_query():
    """
    Body  : { "query": "What is Qatl-i-amd?" }
    Returns: { answer, sources, response_time, query }
    """
    data  = request.get_json()
    query = (data or {}).get('query', '').strip()

    if not query:
        return jsonify({'error': 'Missing query field'}), 400
    if len(query) > 1000:
        return jsonify({'error': 'Query too long (max 1000 chars)'}), 400

    try:
        result = rag_query(query)
        return jsonify(result)
    except Exception as e:
        return jsonify({
            'error'  : str(e),
            'answer' : 'An error occurred. Check Groq API key and connection.',
            'sources': []
        }), 500


# ── GET /api/health ────────────────────────────────────────────────────────
@app.route('/api/health')
def api_health():
    """
    Returns: { groq: ok|error, chromadb: ok|error, model, chunks }
    """
    try:
        count        = vectordb._collection.count()
        chroma_status = 'ok'
    except:
        count, chroma_status = 0, 'error'

    return jsonify({
        'ollama'          : 'ok',           # kept for frontend compatibility
        'groq'            : 'ok',           # Groq API (key loaded = ok)
        'chromadb'        : chroma_status,
        'model'           : GROQ_MODEL,
        'embedding_model' : EMBEDDING_MODEL,
        'chunks'          : count,
    })


# ── GET /api/stats ─────────────────────────────────────────────────────────
@app.route('/api/stats')
def api_stats():
    """
    Returns corpus statistics shown in the sidebar.
    """
    try:
        count = vectordb._collection.count()
    except:
        count = 0

    return jsonify({
        'total_documents' : count,
        'collection_name' : COLLECTION_NAME,
        'embedding_model' : EMBEDDING_MODEL,
        'llm_model'       : GROQ_MODEL,
        'llm_provider'    : 'Groq API',
        'chunk_size'      : 1000,
        'chunk_overlap'   : 200,
        'top_k'           : TOP_K,
        'status'          : 'running'
    })


print('✅ Flask app ready')
print('   GET  /              → index.html')
print('   POST /api/query     → RAG pipeline (Groq)')
print('   GET  /api/health    → system status')
print('   GET  /api/stats     → corpus statistics')

## Step 4 — Launch Server

> 🟡 **Keep this cell running** — it IS the server.
> Open **http://localhost:5000** in your browser.
> Stop with: Kernel → Interrupt

In [ ]:
print('🚀 Starting PakLex AI server...')
print('─' * 50)
print('   App     → http://localhost:5000')
print('   Health  → http://localhost:5000/api/health')
print('   Stats   → http://localhost:5000/api/stats')
print('─' * 50)
print('   LLM     : Groq — mixtral-8x7b-32768')
print('   Embeds  : all-MiniLM-L6-v2 (local)')
print('   VectorDB: ChromaDB (local)')
print('─' * 50)
print('   Press Kernel → Interrupt to stop')

app.run(
    host        = '0.0.0.0',
    port        = 5000,
    debug       = False,
    use_reloader= False,
)

## (Optional) Step 5 — Quick API Test

> Run these in a **separate terminal** while server is active:

In [ ]:
# Run separately from server (e.g. new terminal or separate notebook)
import requests as req, json

# Health
r = req.get('http://localhost:5000/api/health')
print('Health:', json.dumps(r.json(), indent=2))

# Query
r = req.post('http://localhost:5000/api/query',
             json={'query': 'What is the punishment for robbery?'})
res = r.json()
print(f'\nAnswer: {res["answer"][:300]}')
print(f'Time  : {res["response_time"]}s')
print(f'Sources: {len(res["sources"])} chunks')